# Demand shock with sticky wages: minimising the output gap

Monetary policy is chosen to minimise the **output gap** rather than the deviation of
output from its steady state. Two scenarios:

1. **No time to build** — every firm reprices immediately.
2. **Exogenous time to build** — firms reprice with the calibrated lag $\tau$.

The gap is $x_t = y_t - y^n_t$ with $y^n_t = \frac{1+\eta}{\gamma+\eta}a_t$, and the loss is
$\sum_t x_t^2$ (no inflation term). A demand shock shifts the **natural rate**, not
natural output.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools

## 1. Calibration

In [ ]:
sigma = 5
beta = 1 / (1.0 + 0.01)
beta1 = 1 / (1.0 + 0.01 * np.exp(30/(36.5*0.9)))
beta_hh = beta
W = 1

gamma = 2.0    # relative risk aversion / IS slope
eta = 2.0      # inverse Frisch elasticity
PHI_YA = (1 + eta) / (gamma + eta)
rho = 1 / beta_hh - 1

print(f'gamma={gamma}, eta={eta}  ->  IS slope 1/gamma={1/gamma:.4f}, phi_ya={PHI_YA:.4f}')

In [ ]:
data = pd.read_csv('APP_data_usa_goods.csv', index_col=0, parse_dates=True)
data_tau = (data['invtCogsRatio'] * 36.5 / (30 * 0.1))
plt.hist(data_tau, bins=55)
plt.title('Distribution of Inventory Adjustment Times')
plt.xlabel('Adjustment Time (tau)'); plt.ylabel('Frequency')
plt.show()

In [ ]:
tau_diff = np.zeros((1000, 1))
for i, c in enumerate(np.linspace(0, 0.99, 1000)):
    if beta * c <= 1e-10:
        tau_diff[i] = np.inf
        continue
    N = -(data_tau) - 1/np.log(beta * c)
    tau1 = -1/np.log(beta * c) - N;   tau1[tau1 < 0] = 0
    tau2 = -1/np.log(beta1 * c) - N;  tau2[tau2 < 0] = 0
    tau_diff[i] = np.mean(np.abs(tau1 - tau2))

c_list = np.linspace(0, 0.99, 1000)
c = np.min(c_list[np.where(np.abs(tau_diff - 1) == np.min(np.abs(tau_diff - 1)))[0]])
print('optimal c:', c)

In [ ]:
N = -data_tau - 1/np.log(beta * c)
tau = -1/np.log(beta * c) - N
tau[tau < 0] = 0
tau_floor, tau_ceil = np.floor(tau), np.ceil(tau)
tprod_ceil = (beta * c)**tau_ceil * (tau_ceil + N)
tprod_floor = (beta * c)**tau_floor * (tau_floor + N)
tau[(tprod_ceil - tprod_floor) > 0] = tau_ceil[(tprod_ceil - tprod_floor) > 0]
tau[(tprod_ceil - tprod_floor) < 0] = tau_floor[(tprod_ceil - tprod_floor) < 0]
tau[tau < 0] = 0

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(N, bins=50); ax[0].set_title('Distribution of N'); ax[0].set_xlabel('N')
ax[1].hist(tau, bins=55); ax[1].set_title('Distribution of Time to Produce'); ax[1].set_xlabel('tau')
plt.tight_layout(); plt.show()

## 2. The IS curve in gap form

$$x_t = x_{t+1} - \frac{1}{\gamma}\bigl(i_t - \pi_{t+1} - r^n_t\bigr), \qquad x_T = 0$$

$$y^n_t = \frac{1+\eta}{\gamma+\eta}a_t, \qquad
  r^n_t = \rho - d_t + \gamma\,(y^n_{t+1}-y^n_t)$$

The demand shock $d_t$ enters the **natural rate**. With $\rho = 1\%$ and $d_0 = 1\%$ the
two cancel exactly, so $r^n_0 = 0$ up to the time-to-produce term — which is why the
gap-minimising policy cuts to roughly zero in period 0.

In [ ]:
def euler_path_gap_2(i_path, price_level, demand_level, a_nat, gamma=2.0, eta=2.0,
                     beta_hh=1/(1+0.01)):
    """
    IS curve in output-gap form with a demand shock.  Returns (x, pi, y_nat, r_nat).

        x_t   = x_{t+1} - (1/gamma) * ( i_t - pi_{t+1} - r^n_t ),   x_T = 0
        pi_t  = Delta price_level_t
        r^n_t = rho - d_t + gamma * ( y^n_{t+1} - y^n_t )
    """
    i_path = np.asarray(i_path, float)
    price_level = np.asarray(price_level, float)
    demand_level = np.asarray(demand_level, float)
    a_nat = np.asarray(a_nat, float)

    T = len(i_path)
    rho = 1.0 / beta_hh - 1.0
    phi_ya = (1.0 + eta) / (gamma + eta)

    pi = np.zeros(T)
    pi[1:] = price_level[1:] - price_level[:-1]
    pi_fwd = np.r_[pi[1:], 0.0]

    y_nat = phi_ya * a_nat
    dy_nat_fwd = np.r_[y_nat[1:] - y_nat[:-1], 0.0]
    r_nat = rho - demand_level + gamma * dy_nat_fwd

    x = np.zeros(T + 1)
    for t in range(T - 1, -1, -1):
        x[t] = x[t + 1] - (1.0 / gamma) * (i_path[t] - pi_fwd[t] - r_nat[t])

    return x[:T], pi, y_nat, r_nat

In [ ]:
def summarise(label, dev, r_arrange, matches, r_choice, X_gap):
    """Standard summary block for a gap experiment."""
    dev = np.asarray(dev).ravel()
    best = int(np.argmin(dev))
    base = int(np.where(matches)[0][0])
    w = 46
    print('=' * 68)
    print(label)
    print('=' * 68)
    print(f'{"loss  sum x_t^2,  no policy change":<{w}}: {dev[base]:.4e}')
    print(f'{"loss  sum x_t^2,  gap-minimising policy":<{w}}: {dev[best]:.4e}')
    if dev[base] > 1e-20:
        print(f'{"reduction in loss":<{w}}: {100*(1-dev[best]/dev[base]):.1f}%')
    print(f'{"peak |x_t|,  no policy change":<{w}}: {np.max(np.abs(X_gap[base])):.4e}')
    print(f'{"peak |x_t|,  gap-minimising policy":<{w}}: {np.max(np.abs(X_gap[best])):.4e}')
    print(f'{"rate sequence,  no policy change":<{w}}: {np.round(r_arrange[base], 6)}')
    print(f'{"rate sequence,  gap-minimising policy":<{w}}: {np.round(r_arrange[best], 6)}')
    on_bound = bool(np.any(np.isclose(r_arrange[best], r_choice[0]) |
                           np.isclose(r_arrange[best], r_choice[-1])))
    print(f'{"grid":<{w}}: {len(r_choice)} points on '
          f'[{r_choice[0]:g}, {r_choice[-1]:g}]')
    print(f'{"optimum sits on a grid bound":<{w}}: {on_bound}')
    if on_bound:
        print('   -> the optimum may not be interior; widen r_choice to check.')
    return best, base


def gap_figure(label, X_gap, I_path, best, base, fname, horizon=20):
    """Standard 2-panel gap figure: the output gap and the rate path, same horizon."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    t = np.arange(horizon)

    axes[0].plot(t, X_gap[base][:horizon], lw=2, label='no policy change')
    axes[0].plot(t, X_gap[best][:horizon], '--', lw=2, label='gap-minimising policy')
    axes[0].axhline(0, color='k', lw=0.8)
    axes[0].set_xlabel('Time'); axes[0].set_ylabel('$x_t = y_t - y^n_t$')
    axes[0].set_xlim(0, horizon - 1)
    axes[0].set_title('Output gap')
    axes[0].legend(fontsize=8)

    axes[1].step(t, I_path[base][:horizon], where='post', lw=2, label='no policy change')
    axes[1].step(t, I_path[best][:horizon], where='post', linestyle='--', lw=2,
                 label='gap-minimising policy')
    axes[1].set_xlabel('Time'); axes[1].set_ylabel('$i_t$')
    axes[1].set_xlim(0, horizon - 1)
    axes[1].set_title('Nominal interest rate')
    axes[1].legend(fontsize=8)

    fig.suptitle(label, fontsize=13)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

## 3. Gap results: no time to build

No TFP movement, so potential output is flat ($y^n_t = 0$) and the entire shock lands in
the natural rate.

In [ ]:
T = 10
sequence_length = 3
r_choice = np.linspace(0, 0.02, 9)
r_arrange = np.array(list(itertools.product(r_choice, repeat=sequence_length)))
size = r_arrange.shape[0]

deviation_gap = np.zeros(size)
Y_output = np.zeros((size, T))
Y_natural = np.zeros((size, T))
X_gap = np.zeros((size, T))
R_natural = np.zeros((size, T))
I_path_store = np.zeros((size, T))

for k in range(size):
    i_path = np.ones(T) * rho
    i_path[:sequence_length] = r_arrange[k]

    demand = np.zeros(T)
    demand[0] = 0.01                        # 1 percent demand shock

    x, pi, y_nat, r_nat = euler_path_gap_2(i_path, np.zeros(T), demand, np.zeros(T),
                                           gamma=gamma, eta=eta, beta_hh=beta_hh)

    X_gap[k] = x
    Y_natural[k] = y_nat
    Y_output[k] = x + y_nat
    I_path_store[k] = i_path
    R_natural[k] = r_nat
    deviation_gap[k] = np.sum(x**2)

matches = np.all(np.isclose(r_arrange, rho), axis=1)
best_ntb, base_ntb = summarise('1 percent demand shock -- No time to build',
                               deviation_gap, r_arrange, matches, r_choice, X_gap)
print(f'{"natural rate r^n over the free periods":<46}: '
      f'{np.round(R_natural[0][:sequence_length], 6)}')

In [ ]:
gap_figure('1 percent demand shock: No time to build',
           X_gap, I_path_store, best_ntb, base_ntb,
           '../semi_endo_demand_gap_no_time_to_build.png', horizon=10)

## 4. Gap results: exogenous time to build

Firms reprice with the calibrated lag $\tau$. Potential output is no longer flat: the rate
path feeds into firms' discounted production `tprod`, so policy shifts $y^n$ through the
time-to-produce channel. The natural benchmark is the CES productivity index built from
the same `tprod` that generates the price index.

In [ ]:
T = 30
sequence_length = 3
r_choice = np.linspace(0, 0.0125, 6)
r_arrange = np.array(list(itertools.product(r_choice, repeat=sequence_length)))
size = r_arrange.shape[0]

deviation_gap = np.zeros(size)
Y_output = np.zeros((size, T))
Y_natural = np.zeros((size, T))
X_gap = np.zeros((size, T))
R_natural = np.zeros((size, T))
I_path_store = np.zeros((size, T))

tvals = tau.astype(int).to_numpy()
N_array = np.array(N)

for k in range(size):
    n = sequence_length
    r = np.asarray(r_arrange[k], dtype=float)

    interest_rate_list = np.ones((T, T)) * rho
    arr2d = np.full((n, n), rho, dtype=float)
    for i in range(n):
        arr2d[i, :i+1] = r[i::-1]
    interest_rate_list[0:n, :n] = arr2d

    beta_mat = 1.0 / (1.0 + interest_rate_list)
    cumprod_ext = np.ones((T, T + 1))
    cumprod_ext[:, 1:] = np.cumprod(beta_mat, axis=1)
    beta_prod = cumprod_ext[:, tvals]

    tprod_array = np.array((beta_prod * 0.9018018) *
                           ((tvals + N_array) * np.ones((T, 269))))

    A_price = np.sum(1/tprod_array**(sigma - 1), axis=1)**(1/(sigma - 1))   # CES price index
    A_tfp = np.sum(tprod_array**(sigma - 1), axis=1)**(1/(sigma - 1))       # CES productivity

    i_path = np.ones(T) * rho
    i_path[0:sequence_length] = r_arrange[k]

    demand = np.zeros(T)
    demand[0] = 0.01
    a_nat = np.log(A_tfp) - np.log(A_tfp)[-1]

    x, pi, y_nat, r_nat = euler_path_gap_2(i_path, np.log(A_price), demand, a_nat,
                                           gamma=gamma, eta=eta, beta_hh=beta_hh)

    X_gap[k] = x
    Y_natural[k] = y_nat
    Y_output[k] = x + y_nat
    I_path_store[k] = i_path
    R_natural[k] = r_nat
    deviation_gap[k] = np.sum(x**2)

matches = np.all(np.isclose(r_arrange, rho), axis=1)
best_ttb, base_ttb = summarise('1 percent demand shock -- Exogenous time to build',
                               deviation_gap, r_arrange, matches, r_choice, X_gap)
print(f'{"natural rate r^n over the free periods":<46}: '
      f'{np.round(R_natural[best_ttb][:sequence_length], 6)}')

In [ ]:
gap_figure('1 percent demand shock: Exogenous time to build',
           X_gap, I_path_store, best_ttb, base_ttb,
           '../semi_endo_demand_gap_exogenous_time_to_build.png', horizon=10)

## Escalating the policy horizon

Every optimum above sits on a bound of `r_choice`, which means the answer is being shaped
by the grid and by the short window of policy freedom rather than by the economics. This
section grows the number of free periods $L$ — everything after $L$ stays pinned at $\rho$
— and re-solves, stopping when the **last free period comes back to $\rho$**. At that
point the horizon is long enough that pinning the tail no longer binds.

Grid enumeration is $6^L$ and blows up immediately, so this uses a bounded continuous
solve (L-BFGS-B on $[0,\,0.05]$, the lower bound being the ZLB) instead of enumerating.

In [ ]:
from scipy.optimize import minimize


def escalate(objective, T, L_start=3, L_max=16, tol=1e-4, bounds=(0.0, 0.05)):
    """
    Grow the number of free policy periods L until the optimal sequence's LAST free
    period returns to rho.  `objective(r_vec, T)` is MINIMISED (pass -utility for the
    planner problems).  Returns (history, settled_L); settled_L is None if it never
    settles by L_max.
    """
    hist = []
    print(f'{"L":>3} {"last i_t":>10} {"settled":>8} {"objective":>15}   optimal sequence')
    print('-' * 92)
    for L in range(L_start, L_max + 1):
        res = minimize(lambda r: objective(r, T), np.ones(L) * rho,
                       method='L-BFGS-B', bounds=[bounds] * L,
                       options={'maxiter': 20000, 'ftol': 1e-18, 'gtol': 1e-14})
        settled = bool(np.isclose(res.x[-1], rho, atol=tol))
        hist.append((L, res.x.copy(), float(res.fun), settled))
        print(f'{L:>3} {res.x[-1]:>10.6f} {str(settled):>8} {res.fun:>15.6e}   '
              f'{np.round(res.x[:8], 5)}{" ..." if L > 8 else ""}')
        if settled:
            print(f'\n-> settles at L = {L}: the last free period is back at '
                  f'rho = {rho:.4f}')
            return hist, L
    print(f'\n-> NOT settled by L = {L_max}: the optimum is still away from rho.')
    return hist, None


def escalate_figure(hist, settled_L, label, fname):
    """Objective against L, and the rate path at the final L."""
    Ls = [h[0] for h in hist]
    objs = [h[2] for h in hist]
    r_last = hist[-1][1]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(Ls, objs, 'o-')
    axes[0].set_yscale('symlog', linthresh=1e-18)
    axes[0].set_xlabel('free policy periods $L$'); axes[0].set_ylabel('objective')
    axes[0].set_title('Objective as the horizon grows')

    p = np.arange(len(r_last))
    axes[1].step(p, r_last, where='post', lw=2, label=f'optimal $i_t$ at L={Ls[-1]}')
    axes[1].axhline(rho, color='k', lw=0.8, linestyle='--', label=r'$\rho$')
    axes[1].set_xlabel('Period'); axes[1].set_ylabel('$i_t$')
    axes[1].set_title('Optimal rate path at the final $L$')
    axes[1].legend(fontsize=8)

    status = f'settles at L={settled_L}' if settled_L else 'does not settle'
    fig.suptitle(f'{label}  ({status})', fontsize=13)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def price_tfp_from_rates(r, T):
    L = len(r)
    interest_rate_list = np.ones((T, T)) * rho
    arr2d = np.full((L, L), rho, dtype=float)
    for i in range(L):
        arr2d[i, :i+1] = np.asarray(r, float)[i::-1]
    interest_rate_list[0:L, :L] = arr2d
    ce = np.ones((T, T + 1))
    ce[:, 1:] = np.cumprod(1.0/(1.0 + interest_rate_list), axis=1)
    bp = ce[:, tvals]
    tp = (bp * 0.9018018) * ((tvals + N_array) * np.ones((T, 269)))
    A_price = np.sum(1/tp**(sigma - 1), axis=1)**(1/(sigma - 1))
    A_tfp = np.sum(tp**(sigma - 1), axis=1)**(1/(sigma - 1))
    return np.log(A_price), np.log(A_tfp) - np.log(A_tfp)[-1]


def obj_gap_ntb(r, T):
    ip = np.ones(T) * rho; ip[:len(r)] = r
    d = np.zeros(T); d[0] = 0.01
    x, *_ = euler_path_gap_2(ip, np.zeros(T), d, np.zeros(T),
                             gamma=gamma, eta=eta, beta_hh=beta_hh)
    return np.sum(x**2)


def obj_gap_ttb(r, T):
    ip = np.ones(T) * rho; ip[:len(r)] = r
    d = np.zeros(T); d[0] = 0.01
    pl, an = price_tfp_from_rates(r, T)
    x, *_ = euler_path_gap_2(ip, pl, d, an, gamma=gamma, eta=eta, beta_hh=beta_hh)
    return np.sum(x**2)


print('NO TIME TO BUILD')
hist_ntb, L_ntb = escalate(obj_gap_ntb, 10, L_start=3, L_max=9)
print()
print('EXOGENOUS TIME TO BUILD')
hist_ttb, L_ttb = escalate(obj_gap_ttb, 30, L_start=3, L_max=16)

In [ ]:
escalate_figure(hist_ntb, L_ntb, '1% demand shock: no time to build',
                '../semi_endo_demand_gap_escalate_no_time_to_build.png')
escalate_figure(hist_ttb, L_ttb, '1% demand shock: exogenous time to build',
                '../semi_endo_demand_gap_escalate_exogenous_time_to_build.png')